In [2]:
import os
import pandas as pd
from plotly import graph_objects as go
import numpy as np
from Bio import Phylo
import plotly.io as pio
import shutil
import importlib
from tqdm import tqdm
from plotly.subplots import make_subplots
from pathlib import Path
if not Path('jw_utils').exists():
    !git clone https://github.com/JonWinkelman/jw_utils.git
from jw_utils import plotly_preferences as pprefs
from jw_utils import itol_annotations as ia
from jw_utils import ncbi_utils as nu
from jw_utils import parse_fasta as pfa
from jw_utils import hmmer_utils as hu
from jw_utils import parse_gff as pgf
from jw_utils import alignment_utils2 as au

if not Path('orthofinder_utils').exists():
    !git clone https://github.com/JonWinkelman/orthofinder_utils.git
from orthofinder_utils import dash_ortho_parser as dop
from orthofinder_utils import run_orthofinder as ru
import external_functions as ef

In [18]:
AsnC_trans_reg_hmm = 'PF01037'
db_base = Path('/Volumes/Extreme_SSD/bioinformatics_tools/Databases/ncbi_genomes_bacteria_v226/')
db_proteomes = db_base / 'proteomes/'
AsnC_trans_reg_hmm_fp = Path(f'./data/references/{AsnC_trans_reg_hmm}.hmm')
results_dir = Path('results/2_hmmsearch_asnC_moraxellaceae')
results_dir.mkdir(exist_ok=True)

In [48]:
tax_df = pd.read_csv(db_base / 'bac120/bac120_taxonomy_expanded.tsv', sep='\t', index_col=0)
db_proteome_fps = [f for f in db_proteomes.glob('*.faa')]
db_accs = [f.stem for f in db_proteome_fps]
tax_df = tax_df.loc[db_accs]
morax_f    = tax_df['Family'] == 'Moraxellaceae'
acineto_f  = tax_df['Genus'] == 'Acinetobacter'
morax_df   = tax_df[morax_f]
acineto_df = tax_df[acineto_f]

In [22]:
out_dir = results_dir / f'raw_{AsnC_trans_reg_hmm}_search_results'
out_dir.mkdir(exist_ok=True)
for acc, row in morax_df.iterrows():
    out_fp = out_dir / f'{acc}.txt'
    proteome_fp = db_proteomes / f'{acc}.faa'
    hu.run_simple_search(str(AsnC_trans_reg_hmm_fp),str(out_fp), str(proteome_fp))

In [39]:
dfs = []
for acc, row in morax_df.iterrows():
    out_fp = out_dir / f'{acc}.txt'
    df = hu.parse_hmmsearch_output(out_fp)
    df['genome_acc'] = [acc for _ in range(df.shape[0])]
    df['full_ID'] = [f'{acc}_{target}' for acc, target in zip(df['genome_acc'], df['target_name'])]
    df.set_index('full_ID')
    if df.shape[0] > 0:
        dfs.append(df)
hmmsearch_df = pd.concat(dfs)

In [50]:
acineto_df

assembly_accession
RS_GCF_006757745.1    Acinetobacter radioresistens
RS_GCF_000369065.1      Acinetobacter haemolyticus
RS_GCF_000369405.1       Acinetobacter sp000369405
RS_GCF_000369485.1         Acinetobacter dispersus
RS_GCF_029024105.1           Acinetobacter lwoffii
                                  ...             
GB_GCA_008017255.1       Acinetobacter sp008017255
GB_GCA_008017315.1       Acinetobacter sp008017315
GB_GCA_009912575.1      Acinetobacter bereziniae_A
GB_GCA_013530545.1         Acinetobacter lwoffii_G
GB_GCA_018883945.1      Acinetobacter avistercoris
Name: Species, Length: 221, dtype: object